In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

device: cuda
GPU: Tesla T4


In [7]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

data = torch.tensor(
    encode(text),
    dtype=torch.long
)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [15]:
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 10

n_emb = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [17]:
from v8 import GPTLanguageModel

model = GPTLanguageModel(
    vocab_size=vocab_size,
    block_size=block_size,
    n_emb=n_emb,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout
).to(device)

In [18]:
sum(p.numel() for p in model.parameters()) / 1e6

10.788929

In [19]:
def get_batch(split):

    source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [20]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            _, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [24]:
import time

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

start_time = time.time()
last_log_time = start_time

for step in range(max_iters):

    # evaluate occasionally
    if step % eval_interval == 0:
        losses = estimate_loss()

        torch.cuda.synchronize()
        now = time.time()

        elapsed = now - start_time

        # avoid weird ETA at step 0
        if step > 0:
            sec_per_step = elapsed / step
            remaining_steps = max_iters - step
            eta_sec = remaining_steps * sec_per_step

            eta_min = eta_sec / 60
            elapsed_min = elapsed / 60
        else:
            sec_per_step = 0
            eta_min = 0
            elapsed_min = 0

        lr = optimizer.param_groups[0]["lr"]

        print(
            f"[{step:5d}/{max_iters}] "
            f"train {losses['train']:.4f} | "
            f"val {losses['val']:.4f} | "
            f"lr {lr:.2e} | "
            f"{sec_per_step:.3f} s/step | "
            f"elapsed {elapsed_min:.1f} min | "
            f"ETA {eta_min:.1f} min"
        )

    # training batch
    xb, yb = get_batch("train")

    # forward
    logits, loss = model(xb, yb)

    # backward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # update
    optimizer.step()

torch.cuda.synchronize()

total_time = time.time() - start_time

print()
print(f"Training complete")
print(f"Final loss: {loss.item():.4f}")
print(f"Total time: {total_time / 60:.1f} min")
print(f"Average: {total_time / max_iters:.3f} s/step")

[    0/5000] train 4.0166 | val 4.1275 | lr 3.00e-04 | 0.000 s/step | elapsed 0.0 min | ETA 0.0 min
[  500/5000] train 1.6240 | val 1.7810 | lr 3.00e-04 | 0.540 s/step | elapsed 4.5 min | ETA 40.5 min
[ 1000/5000] train 1.4435 | val 1.6427 | lr 3.00e-04 | 0.536 s/step | elapsed 8.9 min | ETA 35.7 min
[ 1500/5000] train 1.3696 | val 1.5789 | lr 3.00e-04 | 0.534 s/step | elapsed 13.3 min | ETA 31.1 min
[ 2000/5000] train 1.2859 | val 1.5295 | lr 3.00e-04 | 0.533 s/step | elapsed 17.8 min | ETA 26.6 min
[ 2500/5000] train 1.2403 | val 1.5122 | lr 3.00e-04 | 0.532 s/step | elapsed 22.2 min | ETA 22.2 min
[ 3000/5000] train 1.1982 | val 1.5017 | lr 3.00e-04 | 0.532 s/step | elapsed 26.6 min | ETA 17.7 min
[ 3500/5000] train 1.1579 | val 1.4981 | lr 3.00e-04 | 0.531 s/step | elapsed 31.0 min | ETA 13.3 min
[ 4000/5000] train 1.1092 | val 1.4914 | lr 3.00e-04 | 0.531 s/step | elapsed 35.4 min | ETA 8.8 min
[ 4500/5000] train 1.0799 | val 1.5144 | lr 3.00e-04 | 0.530 s/step | elapsed 39.8 min 

In [26]:
eval_iters = 1000

model.eval()
losses = estimate_loss()

final_train_loss = losses["train"]
final_val_loss = losses["val"]

print(f"Final train loss: {final_train_loss:.4f}")
print(f"Final val loss:   {final_val_loss:.4f}")

model.train()

Final train loss: 1.0449
Final val loss:   1.4902


GPTLanguageModel(
  (token_embedding_table): Embedding(65, 384)
  (position_embedding_table): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffw): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=

In [31]:
import sys

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),

    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_emb": n_emb,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,

    "stoi": stoi,
    "itos": itos,

    "step": max_iters,
    "max_iters": max_iters,
    "learning_rate": learning_rate,

    "final_train_loss": 1.0449,
    "final_val_loss": 1.4902,

    "torch_version": str(torch.__version__),
    "python_version": sys.version,
    "device_trained_on": str(next(model.parameters()).device),
}

torch.save(checkpoint, "v8_checkpoint.pt")

In [33]:
import os

print(os.path.exists("v8_checkpoint.pt"))
print(f"{os.path.getsize('v8_checkpoint.pt') / 1024**2:.1f} MB")

True
132.7 MB


In [34]:
with open("v8_info.txt", "w") as f:
    f.write(str(model))
    f.write("\n\n")
    f.write(f"Final train loss: {1.0449}\n")
    f.write(f"Final val loss:   {1.4902}\n")
    f.write(f"Training steps:   {max_iters}\n")
    f.write(f"Learning rate:    {learning_rate}\n")

In [35]:
loaded = torch.load("v8_checkpoint.pt", map_location=device)

model_test = GPTLanguageModel(
    vocab_size=loaded["vocab_size"],
    block_size=loaded["block_size"],
    n_emb=loaded["n_emb"],
    n_head=loaded["n_head"],
    n_layer=loaded["n_layer"],
    dropout=loaded["dropout"],
).to(device)

model_test.load_state_dict(loaded["model_state_dict"])
model_test.eval()

print("Checkpoint loaded successfully")

Checkpoint loaded successfully


In [37]:
model_test.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model_test.generate(
    context,
    max_new_tokens=1000
)

text = decode(generated[0].tolist())

print(text)


Hath call'd it you;
And, by my bawd welcome, for your good worship
To bring the other mischance; for in the usure
Would blush through the king of your queen,
Setting upon the deep pale matter, go off
To conceit with Romans, have King Edward's untimely.
Come, know'st me with are yet. Come, lords,
And DeIV:
As you will not shriek the usurping here.

GLOUCESTER:
So much mine own founds upon that means thou shouldst,
Thou art men's daughter. But Tranio, shall find
Then out of banishment. But thou getterness
Should make good Lady Bona and Dick;
Why, ask you, All-Some hardly she is quickly known;
She shall be given and lend her faces.

Second Lord:
My lord, would have I so? when she had a loss thoughts
I have never know me would not stuck him.
If yet, I till be remember for the while:
If we have been send to enter'd circumsuance.

First Senator:
Now, by my liege, for 'tis comes a which straight,
Wherein they have disguised and nose men
Thither napes nothing, were I pin our tents as home,
Su

In [38]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model_test.generate(
    context,
    max_new_tokens=15000
)

# Remove the initial context character
text = decode(generated[0, 1:].tolist())

with open("v8_sample_15000.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"Saved {len(text)} characters to v8_sample_15000.txt")

Saved 15000 characters to v8_sample_15000.txt
